In [8]:
# Onyxia GPU compatible torch version
import sys
!{sys.executable} -m pip install torch==2.11.0 torchaudio torchvision \
    --index-url https://download.pytorch.org/whl/cu126 \
    --force-reinstall

Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 95.9 MB/s  0:00:07:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 97.8 MB/s  0:00:03ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 103.4 MB/s  0:00:0200:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 90.7 MB/s  0:00:01ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 11.6 MB/s  0:00:24:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 166.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 22.3 MB/s  0:00:09:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 28.5 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.7 MB/s  0:00:03 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 2

# Topic modeling

Here is our full implementation of a BERTopic unsupervised model applied to biathlon. 
*Please note that if you try to run the code, the topics described in this notebook might change of index.*

Needed libraries.

In [6]:
!pip install bertopic sentence-transformers umap-learn hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 53.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 76.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 36.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 65.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 73.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 83.4 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 65.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 54.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 75.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 47.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 79.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 78.7 MB/s  0:00:00m0:0

Creation of the corpus dataframe.

In [ ]:
data_path = "PESSD/biath/"

df = pd.DataFrame()
for file in os.listdir(data_path) : 
    temp = pd.read_csv(data_path+file)
    df = pd.concat([df,temp])


Small preporcessing : removing non relevant texts, nouns, stopwords and splitting interventions into sentences.

In [17]:
import re
import string

# Removing music and noise (only technical commentary)
df = df[(df["main_g"]=="female")|(df["main_g"]=="male")]

# text to sentences
df["text"] = df["text"].apply(lambda x : x.split(". "))
df = df.explode("text")

# Removes all words with capitals
df["text"] = df["text"].apply(lambda x : re.sub(r"\s*[A-Z]\w*\s*", " ", x).strip())

# Removes empty texts
df = df[df["text"].str.strip() != ""]

# Remove punctuation
df ["text"] = df["text"].apply(lambda x : x.translate(str.maketrans('', '', string.punctuation.replace("'", ""))))

Adding metadata of the files (gender of the athletes).

In [18]:
df["ID"] = df["audio_file"].str.replace("_wav.wav","")
df = df.merge(pd.read_csv("metadata.csv"), on="ID")
df

,start,stop,text,main_speaker,main_g,audio_file,ID,g_ath
0,0.0,24.0,'est le rêve avec 4 comme le nombre de ...,SPEAKER_02,male,BMF_wav.wav,BMF,F
1,0.0,24.0,l'athlète déjà la plus médaillée chez les fem...,SPEAKER_02,male,BMF_wav.wav,BMF,F
2,0.0,24.0,puis titrée il ne faut pas l'oublier sur la ...,SPEAKER_02,male,BMF_wav.wav,BMF,F
3,0.0,24.0,médaille d'or dans l'histoire du biathlon pour...,SPEAKER_02,male,BMF_wav.wav,BMF,F
4,39.0,71.3,elle de la pourte du sprint enfin celle qui a...,SPEAKER_02,male,BMF_wav.wav,BMF,F
...,...,...,...,...,...,...,...,...
8765,2627.1,2642.1,'est une info qui est tombée il y a quelques m...,SPEAKER_02,male,BAP_wav.wav,BAP,F
8766,2642.1,2676.1,réserve de l'état de forme et de santé de,SPEAKER_01,male,BAP_wav.wav,BAP,F
8767,2642.1,2676.1,en tout cas met un peu plus l' dans les étoi...,SPEAKER_01,male,BAP_wav.wav,BAP,F
8768,2642.1,2676.1,qui revenait de blessure c'est un peu la même ...,SPEAKER_01,male,BAP_wav.wav,BAP,F


Unserpervised topic model based on the corpus.
*If using CPU, please remove the "device" argument in* `SentenceTransformer`.

In [19]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
import hdbscan

# Stopwords list
STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

# Docs 
docs = df["text"]

# Vectorizer for topic representation
vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(1,2),
    min_df=2
    #token_pattern=r"\b(?!\w*[A-Z])\w+\b" # theoreticaly removes words with capital letters once embeddings are done
)

# To control for number and size of clusters
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=50,      # high for larger topics
    min_samples=10,
    cluster_selection_epsilon=0.1,
    cluster_selection_method='leaf',
    prediction_data=True
)

# Load CamemBERT embedding model
embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device="cuda") # remove device if CPU

# Influence choice of themes
seed =  [
["émotion","larme","ému","joie","tristesse"],
["famille","parents","ami", "personnel", "conjoint"]]

# Build BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=None    
)

# Fit the model
topics, probs = topic_model.fit_transform(docs)

# Show discovered topics
# print(topic_model.get_topic_info())

# Visualize topic hierarchy
topic_model.visualize_hierarchy()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3932.80it/s]
2026-04-28 19:13:49,685 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 275/275 [00:10<00:00, 25.05it/s]
2026-04-28 19:14:00,721 - BERTopic - Embedding - Completed ✓
2026-04-28 19:14:00,722 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-28 19:14:37,105 - BERTopic - Dimensionality - Completed ✓
2026-04-28 19:14:37,107 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-28 19:14:38,769 - BERTopic - Cluster - Completed ✓
2026-04-28 19:14:38,777 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-28 19:14:38,949 - BERTopic - Representation - Completed ✓


One can see that around 39% of the corpus is not attributed to a topic (3412/8770). The attributed texts are rather uniformously distributed across topics.
The names of the topic also indicates one potential empty topic (6).

In [20]:
topic_model.get_topic_info()[["Topic", "Name","Count"]]

,Topic,Name,Count
0,-1,-1_secondes_sprint_individuel_monde,3427
1,0,0_tir_tir tir_tireur_tirs,309
2,1,1_position_10_tête_10 10,289
3,2,2_secondes_10 secondes_avance_secondes avance,286
4,3,3_attention_aïe_envoie_également,247
5,4,4_skis_ski_temps ski_rapide,238
6,5,5_coucher_réussite_pioche_faute,195
7,6,6____,189
8,7,7_médaille_médailles_argent_médaille argent,177
9,8,8_équipe_filles_athlètes_groupe,162


Adding topics and topic probabilites to the corpus dataframe.

In [21]:
df["topic"] = topics
df["topic_prob"] = probs.max(axis=1)

One can now check the content of each topic by file (ie discipline or moment of the competition).

In [23]:
# Sentences of topic i and by sport
for _, j in df[df["topic"]==4][["text","ID"]].iterrows():
    print(j["ID"],"_", j["text"])

BMF _ voilà c'est en terre selva donc c'est des conditions qu'il faut pousser tout le temps les skis
BMF _ toute façon ces mastarques quand on n'est pas bien en ski ça peut vraiment être compliqué
BMF _ le voyez sur cette neige qui a un petit peu changé
BMF _ fois qu'il y a des compressions comme ça ça bouchonne on est obligé d'écarter les skis pour monter
BMF _ est dans les skis  
BMF _ bonnes skieuses      
BMF _ 'est des fois difficile de tenir le ski
BMF _ les filles allez allez là c'est maintenant que ça va commencer il faut resserrer les boulons il ne faut rien laisser et qui regarde la course
BMF _ et   c'est des skieuses
BMF _ ça doit skier pour
BMF _ est capable de skier vite également
BMF _ il neige ça doit ralentir un peu les conditions
BMF _ qui a été la meilleure skieuse du siècle
BMF _  elle est très très très rapide sur les skis on l'avait vu tout à l'heure avec
BMF _ est impressionnante sur les skis là
BMF _ essayer de se mettre dans les skis
BMF _ dans la tempête de ne

One can also check the main words shaping a topic. Here related to the speed of the athletes.

In [26]:
topic_model.get_topic(4)

[('skis', np.float64(0.1650422447678318)),
 ('ski', np.float64(0.109229568794357)),
 ('temps ski', np.float64(0.05525175079606928)),
 ('rapide', np.float64(0.048090256915924244)),
 ('rapide skis', np.float64(0.04189043173466498)),
 ('neige', np.float64(0.03953635556318862)),
 ('skier', np.float64(0.03487196450558766)),
 ('vite', np.float64(0.03211793921214331)),
 ('vite skis', np.float64(0.031830384343472004)),
 ('temps', np.float64(0.028621437690388506))]

Topic 6 is indeed empty and can be removed of the corpus.

In [25]:
# Sentences of topic i and by sport
for _, j in df[df["topic"]==6][["text","ID"]].iterrows():
    print(j["ID"],"_", j["text"])

df = df[df["topic"]!=6]

BMF _ 
BMF _  
BMF _ 
BMF _ 
BMF _   
BMF _ 
BMF _ 
BMF _ 
BMF _ 
BMF _ 
BMF _ 
BMF _  
BI _ 
BI _ 
BI _ 
BI _ 
BI _  
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _   
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _  
BI _ ' 
BI _ 
BI _    
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BI _ 
BAH _ 
BAH _ 
BAH _ 
BAH _ 
BAH _ 
BAH _ 
BAH _ 
BAH _ 
BAH _ 
BAH _ 
BAH _ 
BAH _ 
BAH _ 
BAH _  
BAH _ 
BAH _ 
BAH _ 
BFR _ 
BFR _  
BFR _ 
BFR _  
BFR _ 
BFR _ 
BFR _ 
BFR _ 
BFR _ 
BFR _ 
BFR _ 
BFR _ 
BFR _ 
BFR _ 
BFR _ 
BFR _ 
BHR _ nouveau encore
BHR _ 
BHR _ 
BHR _ 
BHR _ 
BHR _ 
BHR _ 
BHR _ 
BHR _ 
BHR _ 
BHR _ 
BHR _ 
BHR _ 
BHR _ à quand même
BHR _  
BHR _  
BH _  
BH _ 
BH _ 
BH _ à  '
BH _ 
BH _ 
BH _ 
BH _ 
BIR1 _ 
BIR1 _  
BIR1 _ 
BIR1 _ 
BIR1 _ 
BSF _ 
BSF _ 
BSF _ 
BSF _ 
BSF _ 
BSF _ 
BSF _ 
BSF _  
BSF _ 
BSF _  
BSF _ 
BSF _ 
BSF _ 
BSF _ 
BIF _ 
BIF _ 
BIF _ 
BIF _ 
BIF _ 
BIF _ 
BIF _ 
BIF _ 
BIF _  
BIF _ 
BIF _ 

To summerise the information, the share of commentary on women(*F*), men (*H*) and both (*M*) are computed by topic.

In [27]:
# Only classified docs
clf = df[df["topic"]!=-1]
cdf = pd.DataFrame()
for i in range(len(clf["topic"].unique())) :
    tmp = df[df["topic"]==i]
    tmp = tmp["g_ath"].value_counts(normalize=True)
    tmp = pd.DataFrame([{
    "topic": f"topic_{i}",
    "M": tmp.get("M", 0),
    "F": tmp.get("F", 0),
    "H": tmp.get("H", 0)
}])
    cdf = pd.concat([cdf,tmp])

# All topics
tmp = clf["g_ath"].value_counts(normalize=True)
tmp = pd.DataFrame([{
    "topic": "all_topics",
    "M": tmp.get("M", 0),
    "F": tmp.get("F", 0),
    "H": tmp.get("H", 0)
}])
cdf = pd.concat([cdf,tmp])

for col in ["M", "F", "H"]:
    ref = tmp[col]
    cdf[f"{col}_diff"] = cdf[col] - ref
    cdf[f"{col}_diff"] = cdf[f"{col}_diff"].apply(
        lambda x: f"{x*100:+.1f}%" if pd.notna(x) else ""
    )

cdf


,topic,M,F,H,M_diff,F_diff,H_diff
0,topic_0,0.064725,0.401294,0.533981,-4.1%,+2.7%,+1.3%
0,topic_1,0.100346,0.387543,0.512111,-0.5%,+1.4%,-0.8%
0,topic_2,0.104895,0.395105,0.500000,-0.1%,+2.1%,-2.1%
0,topic_3,0.113360,0.376518,0.510121,+0.8%,+0.3%,-1.0%
0,topic_4,0.117647,0.378151,0.504202,+1.2%,+0.4%,-1.6%
0,topic_5,0.174359,0.394872,0.430769,+6.9%,+2.1%,-9.0%
0,topic_6,0.000000,0.000000,0.000000,-10.6%,-37.4%,-52.1%
0,topic_7,0.079096,0.322034,0.598870,-2.6%,-5.2%,+7.8%
0,topic_8,0.148148,0.345679,0.506173,+4.3%,-2.8%,-1.4%
0,topic_9,0.160256,0.333333,0.506410,+5.5%,-4.1%,-1.4%


Topic 12, 22, 27 and 36 show the biggest gap with the gender shares across all topics.
12 seems to be linked to the relay disciplines, 22 to the compliments on the performance, 27 on the efforts and intensity of the athletes and 36 on taking the lead.

In [32]:
print(topic_model.get_topic(12))
print(topic_model.get_topic(22))
print(topic_model.get_topic(27))
print(topic_model.get_topic(36))

[('relais', np.float64(0.17554919592197524)), ('relayeur', np.float64(0.07280344281807817)), ('mixte', np.float64(0.06504694335990772)), ('relais relais', np.float64(0.04973246080475016)), ('histoire', np.float64(0.04740459846003804)), ('relais mixte', np.float64(0.042385284153419744)), ('zone mixte', np.float64(0.041443717337291804)), ('micro', np.float64(0.03778640158801846)), ('zone', np.float64(0.03578323695473153)), ('relayeurs', np.float64(0.03230348097160908))]
[('course', np.float64(0.25431195674014945)), ('course course', np.float64(0.11121267394453493)), ('belle course', np.float64(0.09395073485748226)), ('sacrée course', np.float64(0.0662802682638889)), ('courses', np.float64(0.06489073297215889)), ('courir', np.float64(0.06127102403606209)), ('couru', np.float64(0.05871920928592641)), ('sacrée', np.float64(0.056567447757796534)), ('mastarte', np.float64(0.05580488079126336)), ('belle', np.float64(0.0546948819663203))]
[('vraiment', np.float64(0.019206207217509028)), ('faut'

Looking more in depth into the topics, it appears that the 27th, though quite noisy, shows the thema of resilience ("effort continu", "se remobiliser", "ne pas craquer"...), which is found to be especially feminine in the literature as in our data.

Topic 28, about emotions, is also more feminine though in a lesser magnitude.

In [38]:
for _, j in df[df["topic"]==27][["text","ID"]].iterrows():
    print(j["ID"],"_", j["text"])

BMF _ sont parties prud'hommes je trouve que là on sent le relâchement sur les skis elles en mettent mais voilà ça reste un effort qui est maîtrisé là pour les françaises
BMF _ selon les configurations du tir si il y en a une ou deux qui partent seules en tête c'est là où il va falloir faire un gros temps de ski pour creuser mais là pour l'instant ça sert à rien de s'exciter
BMF _ là ça neige pas mal et ça temporise quand même cette bosse on voit que l'effort ne sont pas à 100 là les filles parce que derrière  elle reprend du temps
BI _ ouais voilà c'est une grosse pénalité après aujourd'hui c'est quand même une piste pour les gros skieurs parce que parce qu'on vous le détaillera un peu plus tard mais il y a vraiment un effort continu à fournir tout au long des quatre kilomètres qu'ils auront à effectuer cinq fois  vous aurez le 38 de   donc leader du classement général de la du  leader du petit globe de l'individuel grand favori même s'il a encore en face de lui du beau monde
BI _ vou

To see if the topics are consistent, we run the model without 20% of the corpus.
*GPU here is mandatory.*

In [39]:
# Main words of current model themes

reference = set()
for topic_id in topic_model.get_topics():
    if topic_id == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(topic_id)[:10]]
    reference.add(tuple(sorted(words)))

10 runs at 80%. Stability is measured with the Jaccard Index between reference model topics and 80% runs topics. If the index is higher than 0.2 7 times or more, the topic is considered stable.

In [41]:
from sklearn.utils import resample
from collections import Counter

# List of words for each topic for each model
all_topics_words = []

# Number of runs
nb_iter = 10

for i in range(nb_iter):

    # First, resample 80% of docs
    sample = resample(docs, n_samples=int(len(docs) * 0.8), random_state=i)

    # Same model specification
    model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=seed    
)
    model.fit(sample)
    
    # Top 10 words per topic
    top_words = set()
    for topic_id in model.get_topics():
        if topic_id == -1:
            continue
        words = [w for w, _ in model.get_topic(topic_id)[:10]]
        top_words.add(tuple(sorted(words)))
    
    all_topics_words.append(top_words)


# Jaccard similarity index
def jaccard(t1, t2):
    s1, s2 = set(t1), set(t2)
    return len(s1 & s2) / len(s1 | s2)

# Topic stability
"""
Measures similarity between reference topic and topics found in other runs.

Thresholds for stability are determined by Jaccard index (threshold) and number of occurences (min_run)
"""
def is_stable(topic, all_runs, threshold=0.2, min_runs=7):
    count = 0
    for run_topics in all_runs:
        if any(jaccard(topic, t) >= threshold for t in run_topics):
            count += 1
    return count >= min_runs


stable_topics = [t for t in reference if is_stable(t, all_topics_words)]

print(f"Stable topics (≥7 runs out of 10) : {len(stable_topics)}")
for t in stable_topics:
    print(t)


2026-04-28 19:44:32,097 - BERTopic - Embedding - Transforming documents to embeddings.
Batches:   0%|          | 0/220 [00:00<?, ?it/s]

Batches: 100%|██████████| 220/220 [00:08<00:00, 24.81it/s]
2026-04-28 19:44:41,034 - BERTopic - Embedding - Completed ✓
2026-04-28 19:44:41,036 - BERTopic - Guided - Find embeddings highly related to seeded topics.
Batches: 100%|██████████| 1/1 [00:00<00:00, 48.55it/s]
2026-04-28 19:44:41,195 - BERTopic - Guided - Completed ✓
2026-04-28 19:44:41,196 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-28 19:45:23,074 - BERTopic - Dimensionality - Completed ✓
2026-04-28 19:45:23,076 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-28 19:45:24,216 - BERTopic - Cluster - Completed ✓
2026-04-28 19:45:24,226 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-28 19:45:24,391 - BERTopic - Representation - Completed ✓
2026-04-28 19:45:24,678 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 220/220 [00:08<00:00, 25.16it/s]
2026-04-28 19:45:33,485 - BERTopic - Embeddin

Stable topics (≥7 runs out of 10) : 18
('10', '10 10', '3e', 'classement', 'classement général', 'général', 'place', 'position', 'ressortir', 'tête')
('belle', 'belle course', 'courir', 'course', 'course course', 'courses', 'couru', 'mastarte', 'sacrée', 'sacrée course')
('champion', 'champion olympique', 'championne', 'championne olympique', 'médaille', 'olympique', 'olympique titre', 'olympiques', 'titre', 'titre olympique')
('', '', '', '', '', '', '', '', '', '')
('petit temps', 'relance', 'remet', 'remettre', 'reprend', 'reprendre', 'repris', 'repris temps', 'revenir', 'temps')
('droit erreur', 'déjà faute', 'erreur', 'erreurs', 'faute', 'faute fautes', 'fautes', 'fautes fautes', 'parti', 'parti faute')
('boss', 'loin', 'loin loin', 'rentrer', 'rentré', 'ressort', 'ressortir', 'sortie', 'sortir', 'train rentrer')
('15', '15 15', '18', '18 20', '19', '19 20', '20', '20 19', '20 20', 'aller chercher')
('côté', 'français', 'norvège', 'norvégien', 'norvégienne', 'norvégiens', 'suédois